Gigel created the validation dataset with help from an LLM. However, before that he tried doing it on his own (but he got bored). This was his progress:

In [11]:
from PIL import Image
import os
img = Image.open("../../training_data/train_img_01.jpg") #open an image
tiles_folder = "train_tiles"
os.makedirs(tiles_folder, exist_ok=True)

tile = img.crop((0, 0, 128, 128)) #get a tile from image
tile_path = os.path.join(tiles_folder, "img1_tile0.jpg") 
tile.save(tile_path) #save to path

tile = img.crop((128, 0, 256, 128)) #once more
tile_path = os.path.join(tiles_folder, "img1_tile1.jpg")
tile.save(tile_path)

In [12]:
input_folder = "../../training_data"
output_folder = "train_tiles"

os.makedirs(output_folder, exist_ok=True)

tile_size = 128

for img_name in os.listdir(input_folder):
    if not img_name.endswith(".jpg"):
        continue
        
    img_path = os.path.join(input_folder, img_name) #asa le pregatesc din folder mare
    img = Image.open(img_path) #citesc imaginea cu PIL Image

    img_id = img_name.split(".")[0] #numele imaginii fara jpg

    tile_id = 0
    
    for row in range(4):
        for col in range(4):
            left = col * tile_size
            top = row * tile_size
            right = left + tile_size
            bottom = top + tile_size

            tile = img.crop((left, top, right, bottom))

            tile_name = f"{img_id}_tile_{tile_id}.jpg"
            tile.save(os.path.join(output_folder, tile_name))

            tile_id += 1

In [13]:
def parse_tile_name(name):
    if "_tile_" not in name:
        return None, None

    parts = name.split("_tile_")
    img_id = parts[0]
    tile_id = int(parts[1].split(".")[0])
    return img_id, tile_id

In [ ]:
def get_label(tile1, tile2):
    img1, id1 = parse_tile_name(tile1)
    img2, id2 = parse_tile_name(tile2)

    if img1 != img2:
        return 0  # diferite

    r1, c1 = id1 // 4, id1 % 4
    r2, c2 = id2 // 4, id2 % 4

    # vecini
    if r1 == r2 and c1 + 1 == c2:
        return 1  # left
    if r1 + 1 == r2 and c1 == c2:
        return 2  # above
    if r1 == r2 and c1 - 1 == c2:
        return 3  # right
    if r1 - 1 == r2 and c1 == c2:
        return 4  # under

    return 5  # same image - nu diferite

In [ ]:
import random

tiles = [t for t in os.listdir("train_tiles") if "_tile_" in t]

pairs = []

for _ in range(5000):

    if random.random() < 0.5:
        # imagini la fel
        t1 = random.choice(tiles)
        img_id, _ = parse_tile_name(t1)

        same_tiles = [t for t in tiles if img_id in t]
        t2 = random.choice(same_tiles)

    else:
        # imagini diferite
        t1, t2 = random.sample(tiles, 2)

        # ca sa fie diferite
        while parse_tile_name(t1)[0] == parse_tile_name(t2)[0]:
            t2 = random.choice(tiles)

    label = get_label(t1, t2)
    pairs.append((t1, t2, label))

In [16]:
import pandas as pd

df = pd.DataFrame(pairs, columns=["tile1", "tile2", "label"])
df.to_csv("train_pairs.csv", index=False)

Bam! now you have your brand new dataset with a whopping 2 pieces of data!

Also here is the incredible resnet Gigel heard about!

In [17]:
from torchvision import models
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) 
print(resnet)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

If for whatever reason, you don't want some parameters of the model to change you can do this

In [18]:
#freeze all parameters
for param in resnet.parameters():
    param.requires_grad = False 

#if for whatever reason you want the parameters of layer1, block 0 first conv  (this is purely for example purposes, it was chosen randomly)
#to be unfreezed (model can learn and change these parameters) this is how you would do it

for param in resnet.layer1[0].conv1.parameters():
  param.requires_grad = True

Gigel also provided you with a way to generate the output DF!

In [ ]:
# import pandas as pd
# test_df = pd.read_csv("test/test_pairs.csv")
# #len(test_df)
# subtaskID = [1] * len(test_df) + [2] * len(test_df)
# datapointID = test_df['datapointID'].to_list() * 2

# #REPLACE THIS WITH ACTUAL ANSWERS FOR SUBTASK 1
# subtask1Answers = [0] * len(test_df)
# #REPLACE THIS WITH ACTUAL ANSWERS FOR SUBTASK 2
# subtask2Answers = [0] * len(test_df)


# answer = subtask1Answers + subtask2Answers

# output_df = pd.DataFrame({
#     "subtaskID": subtaskID, 
#     "datapointID": datapointID, 
#     "answer": answer
# })
# output_df.to_csv("output.csv")

: 